# 3. Data Preparation

## Objective
Clean, preprocess, and prepare the training data for model training. Follow guidelines from DATA.md.

### Data Processing Steps:
1. Load and combine original training data with external Akkademia data
2. Remove modern scribal notations (!, ?, /, :, etc.)
3. Replace/remove special characters and brackets
4. Normalize Unicode characters
5. Handle gaps and breaks
6. Tokenize for model input
7. Split into train/validation sets

In [ ]:
# Parameters and Paths
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

OUTPUT_DIR = Path("output")
PREPROCESSED_DIR = OUTPUT_DIR / "pre_processed_data"
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

EXTERNAL_DATA_DIR = OUTPUT_DIR / "external_data"
DATA_DIR = Path("data")
print(f"Preprocessed data directory: {PREPROCESSED_DIR}")
print(f"External data directory: {EXTERNAL_DATA_DIR}")

In [ ]:
# Load training data
train_df = pd.read_csv(DATA_DIR / "train.csv")
print(f"Original training data shape: {train_df.shape}")
train_df.head(2)

In [ ]:
# Load external Akkademia data if available
akkademia_corpus_path = EXTERNAL_DATA_DIR / "akkademia_parallel_corpus.csv"

if akkademia_corpus_path.exists():
    akkademia_df = pd.read_csv(akkademia_corpus_path)
    print(f"Loaded Akkademia corpus: {akkademia_df.shape}")
    print(f"Columns: {akkademia_df.columns.tolist()}")
    
    # Sample to check the data
    print("\nSample Akkademia entries:")
    for i in range(min(3, len(akkademia_df))):
        print(f"\nAkkadian: {akkademia_df['transliteration'].iloc[i][:100]}...")
        print(f"English: {akkademia_df['translation'].iloc[i][:100]}...")
    
    # Add source column if not present
    if 'source' not in akkademia_df.columns:
        akkademia_df['source'] = 'akkademia'
    
    # Standardize column names
    akkademia_df = akkademia_df.rename(columns={
        'transliteration': 'transliteration',
        'translation': 'translation'
    })
    
    print(f"\nAkkademia data loaded successfully: {len(akkademia_df)} sentence pairs")
else:
    print(f"Akkademia corpus not found at {akkademia_corpus_path}")
    print("Will proceed with original training data only.")
    akkademia_df = pd.DataFrame()

In [ ]:
# Combine training data with Akkademia data
if len(akkademia_df) > 0:
    # Ensure both dataframes have the same columns
    train_df['source'] = 'train.csv'
    
    # Combine datasets
    combined_df = pd.concat([train_df, akkademia_df], ignore_index=True)
    print(f"Combined data shape: {combined_df.shape}")
    print(f"\nData sources:")
    print(combined_df['source'].value_counts())
else:
    train_df['source'] = 'train.csv'
    combined_df = train_df
    print(f"Using original training data only: {combined_df.shape}")

## 3.1 Text Cleaning Functions

Following the formatting suggestions from DATA.md:
- Remove modern scribal notations (!, ?, /, :)
- Replace gaps with markers
- Handle brackets
- Normalize special characters (Ḫ→H, etc.)

In [ ]:
def clean_transliteration(text):
    """
    Clean Akkadian transliteration following DATA.md guidelines.
    
    Removal rules:
    - ! (certain reading)
    - ? (questionable reading)
    - / (line divider)
    - : OR . (word divider)
    - ˹ ˺ (partially broken signs)
    - [ ] (from document level transliteration)
    
    Replacement rules:
    - [x] → <gap>
    - … → <big_gap>
    - [... …] → <big_gap>
    - ki {ki} → handled specially
    - il5 il5 → normalized
    - Ḫ → H, ḫ → h (test data has only H h)
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Replace gaps and breaks first
    text = re.sub(r'\[x\]', '<gap>', text)
    text = re.sub(r'\.{3,}', '<big_gap>', text)
    text = re.sub(r'\[…\s*…\]', '<big_gap>', text)
    text = re.sub(r'\[\s*\]', '', text)  # Empty brackets
    
    # Remove/modify special characters
    # Remove: ! ? / : . 
    text = re.sub(r'[!?/:.]', '', text)
    
    # Remove partially broken signs
    text = re.sub(r'[˹˺]', '', text)
    
    # Remove square brackets (but keep content)
    text = re.sub(r'\[|\]', '', text)
    
    # Handle subscripted numbers: il5 → il₅, anšen → ANŠE
    # Normalize subscript numbers ₀-₉
    text = re.sub(r'([a-zA-Z])([0-9])', r'\1₍\2₎', text)
    
    # Normalize Ḫ → H and ḫ → h (important: test data only has H/h)
    text = text.replace('Ḫ', 'H').replace('ḫ', 'h')
    text = text.replace('Ṣ', 'S').replace('ṣ', 's')
    text = text.replace('Ṭ', 'T').replace('ṭ', 't')
    
    # Handle special characters
    text = text.replace('Š', 'SZ').replace('š', 'sz')
    
    # Clean up extra whitespace
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text


def clean_translation(text):
    """Clean English translation text"""
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Remove quotes and extra punctuation
    text = text.replace('"', '').replace("''", '')
    
    # Clean up extra whitespace
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

# Test the cleaning functions
sample_text = combined_df['transliteration'].iloc[0]
print("Original:", sample_text[:200])
print("\nCleaned:", clean_transliteration(sample_text)[:200])

In [ ]:
# Apply cleaning to combined training data
combined_df['transliteration_cleaned'] = combined_df['transliteration'].apply(clean_transliteration)
combined_df['translation_cleaned'] = combined_df['translation'].apply(clean_translation)

print("Data cleaning complete!")
print(f"\nSample cleaned transliteration:")
print(combined_df['transliteration_cleaned'].iloc[0][:200])
print(f"\nSample cleaned translation:")
print(combined_df['translation_cleaned'].iloc[0][:200])

## 3.2 Data Validation

In [ ]:
# Check for empty or very short texts
print("=== Data Quality Checks ===")

# Check for empty transliterations
empty_translit = combined_df['transliteration_cleaned'].str.len() == 0
print(f"Empty transliterations: {empty_translit.sum()}")

# Check for empty translations
empty_trans = combined_df['translation_cleaned'].str.len() == 0
print(f"Empty translations: {empty_trans.sum()}")

# Remove empty or very short entries
min_translit_len = 5
min_trans_len = 5

combined_df = combined_df[
    (combined_df['transliteration_cleaned'].str.len() >= min_translit_len) &
    (combined_df['translation_cleaned'].str.len() >= min_trans_len)
]

print(f"\nAfter filtering: {len(combined_df)} samples")
print(f"\nData sources after cleaning:")
print(combined_df['source'].value_counts())

## 3.3 Prepare Test Data

In [ ]:
# Load and process test data
test_df = pd.read_csv(DATA_DIR / "test.csv")
test_df['transliteration_cleaned'] = test_df['transliteration'].apply(clean_transliteration)

print(f"Test data shape: {test_df.shape}")
print(f"\nSample cleaned test transliteration:")
print(test_df['transliteration_cleaned'].iloc[0][:200])

## 3.4 Train/Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split into train and validation, stratify by source if possible
train_data, val_data = train_test_split(
    combined_df, 
    test_size=0.1,  # 10% for validation
    random_state=42
)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_df)}")
print(f"\nTraining data sources:")
print(train_data['source'].value_counts())

## 3.5 Save Processed Data

In [ ]:
# Save processed datasets

# Full processed training data (combined with external)
combined_df.to_csv(PREPROCESSED_DIR / "train_processed.csv", index=False)

# Train/val split
train_data.to_csv(PREPROCESSED_DIR / "train_split.csv", index=False)
val_data.to_csv(PREPROCESSED_DIR / "val_split.csv", index=False)

# Test data
test_df.to_csv(PREPROCESSED_DIR / "test_processed.csv", index=False)

print("Processed data saved:")
for f in PREPROCESSED_DIR.iterdir():
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

## 3.6 Create Parallel Corpus Format

For model training, create a simple parallel corpus format that can be used with various translation frameworks.

In [ ]:
# Create parallel corpus files (one line per sentence pair)
# Format: source ||| target

def create_parallel_corpus(df, source_col, target_col, output_path):
    """Create parallel corpus file"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            source = str(row[source_col]).strip()
            target = str(row[target_col]).strip()
            if source and target:
                f.write(f"{source} ||| {target}\n")

# Create train, val, and test corpus files
create_parallel_corpus(train_data, 'transliteration_cleaned', 'translation_cleaned', 
                       PREPROCESSED_DIR / "train.corpus")
create_parallel_corpus(val_data, 'transliteration_cleaned', 'translation_cleaned', 
                       PREPROCESSED_DIR / "val.corpus")

# For test, just write source
with open(PREPROCESSED_DIR / "test.source", 'w', encoding='utf-8') as f:
    for text in test_df['transliteration_cleaned']:
        f.write(f"{str(text).strip()}\n")

print("Corpus files created:")
for f in PREPROCESSED_DIR.iterdir():
    if f.suffix in ['.corpus', '.source']:
        print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

## 3.7 Create JSONL Format for Transformers

In [ ]:
# Create JSONL format for transformer models

def create_jsonl(df, source_col, target_col, output_path):
    """Create JSONL file for transformer training"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            source = str(row[source_col]).strip()
            target = str(row[target_col]).strip()
            if source and target:
                json_line = json.dumps({
                    'translation': {
                        'akkadian': source,
                        'english': target
                    }
                }, ensure_ascii=False)
                f.write(json_line + '\n')

create_jsonl(train_data, 'transliteration_cleaned', 'translation_cleaned',
            PREPROCESSED_DIR / "train.jsonl")
create_jsonl(val_data, 'transliteration_cleaned', 'translation_cleaned',
            PREPROCESSED_DIR / "val.jsonl")

# Test JSONL
with open(PREPROCESSED_DIR / "train.jsonl", 'r') as f:
    sample = json.loads(f.readline())
print("Sample JSONL entry:")
print(json.dumps(sample, indent=2, ensure_ascii=False))

## 3.8 Save Processing Summary

In [ ]:
# Save processing summary
processing_summary = {
    "original_train_samples": 1561,
    "akkademia_samples": len(akkademia_df) if len(akkademia_df) > 0 else 0,
    "combined_samples": len(combined_df),
    "train_samples": len(train_data),
    "val_samples": len(val_data),
    "test_samples": len(test_df),
    "cleaning_steps": [
        "Removed modern scribal notations (!, ?, /, :, .)",
        "Replaced gap markers [...], … with <gap>/<big_gap>",
        "Removed partially broken signs ˹ ˺",
        "Normalized Ḫ→H, ḫ→h (test data compatibility)",
        "Normalized Ṣ→S, ṭ→t",
        "Cleaned extra whitespace"
    ],
    "data_sources": {
        "train.csv": "Original training data",
        "akkademia": "Akkademia/ORACC RINAP parallel corpus (downloaded from GitHub)"
    },
    "min_transliteration_length": 5,
    "min_translation_length": 5,
    "validation_split": 0.1
}

with open(PREPROCESSED_DIR / "processing_summary.json", 'w') as f:
    json.dump(processing_summary, f, indent=2)

print("Processing summary:")
print(json.dumps(processing_summary, indent=2))

In [ ]:
# Final statistics
print("=== Final Data Statistics ===")
print(f"\nTraining set:")
print(f"  - Samples: {len(train_data)}")
print(f"  - Avg source length: {train_data['transliteration_cleaned'].str.len().mean():.0f} chars")
print(f"  - Avg target length: {train_data['translation_cleaned'].str.len().mean():.0f} chars")

print(f"\nValidation set:")
print(f"  - Samples: {len(val_data)}")
print(f"  - Avg source length: {val_data['transliteration_cleaned'].str.len().mean():.0f} chars")
print(f"  - Avg target length: {val_data['translation_cleaned'].str.len().mean():.0f} chars")

print(f"\nTest set:")
print(f"  - Samples: {len(test_df)}")
print(f"  - Avg source length: {test_df['transliteration_cleaned'].str.len().mean():.0f} chars")

In [ ]:
# Verify notebook is valid JSON
notebook_path = Path("notebooks/03_data_preparation.ipynb")
if notebook_path.exists():
    with open(notebook_path, 'r') as f:
        nb = json.load(f)
    print(f"Notebook is valid JSON with {len(nb['cells'])} cells")